# Cost function for a pendulum swing-up (DP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/courses/gro860/labs/cost_function_pendulum.ipynb)

This page shows a quick demo of how DP (dynamic programming) can be used for finding a global optimal control policy for a pendulum, with the option to easily modify the cost function to see how it influences the solution.

**GRO860 exercise C.1.5 — Fonction de coût pour un pendule.** For each situation below, run the notebook and analyse (1) the cost-to-go figure $J^*$, (2) the control law map, and (3) the simulated trajectory:

- **a)** Reference situation: run with the default quadratic cost.
- **b)** Position penalty: increase the position weight in the $Q$ matrix.
- **c)** Minimum time: modify $g(x,u,t)$ and $h(x,t)$ to minimize the stabilization time.
- **d)** *(Optional)* Free exploration of other cost-function variants.

This page uses the toolbox [minilink](https://github.com/alx87grd/minilink).

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.costs import CostFunction
from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum
from minilink.planning.policy_synthesis import plotting
from minilink.planning.policy_synthesis.discretizer import StateSpaceGrid
from minilink.planning.policy_synthesis.dp import (
    DynamicProgrammingOptions,
    DynamicProgrammingPlanner,
)
from minilink.planning.problems import PlanningProblem

## Defining a dynamic system model

Here we load an already defined class including all the dynamic equations, and we define the domain (for the state $x = [\theta, \dot\theta]$ and the torque $u$) over which we will generate a controller. The target state $[\theta = -\pi, \dot\theta = 0]$ is the upright position.

In [ ]:
plant = Pendulum()

# State and control input domain
plant.inputs["u"].upper_bound = np.array([+5.0])  # max torque
plant.inputs["u"].lower_bound = np.array([-5.0])  # min torque

plant.state.upper_bound = np.array([+6.0, +6.0])  # max angle and velocity
plant.state.lower_bound = np.array([-6.0, -6.0])  # min angle and velocity

## Defining the cost function

Here we can define a cost function of the type

$$J = \int_{0}^{t_f} g(x, u, t) \, dt + h(x_f, t_f)$$

The class below implements a quadratic running cost with an optional zero-cost zone around the target. **This is the class to modify for tasks b), c), and d) of the exercise.**

In [ ]:
class CustomCostFunction(CostFunction):
    """
    J = int( g(x,u,t) * dt ) + h( x(T) , T )
    """

    def __init__(self):

        self.EPS = 0.1

        # Target state
        self.x_target = np.array([-np.pi, 0.0])

        # Quadratic cost weights
        self.Q = np.diag(np.ones(2))
        self.R = np.diag(np.ones(1))

        # Optional zone of zero cost if ||dx|| < EPS
        self.ontarget_check = False

    def g(self, x, u, t=0.0, params=None):
        """Quadratic additive running cost"""

        # Delta values with respect to target state
        dx = x - self.x_target

        dJ = dx.T @ self.Q @ dx + u.T @ self.R @ u

        # Set cost to zero if on target
        if self.ontarget_check:
            if np.linalg.norm(dx) < self.EPS:
                dJ = 0.0

        return dJ

    def h(self, x, t=0.0, params=None):
        """Terminal cost function with zero value"""

        return 0.0

Here we define the parameters used by the cost function.

In [ ]:
cf = CustomCostFunction()

cf.x_target = np.array([-np.pi, 0.0])  # target (upright position)
cf.Q[0, 0] = 1.0
cf.Q[1, 1] = 1.0
cf.R[0, 0] = 1.0

print("Cost function parameters\n-----------------")
print("Target state:\n", cf.x_target)
print("Q:\n", cf.Q)
print("R:\n", cf.R)

## Synthesizing the "optimal" controller

Here we use library tools that:

1. Discretize the domain of the states and control inputs: the 2D state-space is discretized into a 201 x 201 grid, the torque is discretized into 21 discrete levels, and the time step is 0.05 sec.
2. Use value iteration to compute the optimal cost-to-go and control actions based on the previously defined cost function $g(x,u,t)$, solving the Bellman equation to a tolerance `tol`.
3. Generate a continuous control law by interpolating in the computed discrete solution.

In [ ]:
INF = 300.0  # cost-to-go penalty for going out of the domain

problem = PlanningProblem(plant, x_goal=cf.x_target, cost=cf)

grid = StateSpaceGrid(problem, x_grid_shape=(201, 201), u_grid_shape=(21,), dt=0.05)

Here we use the algorithm called *value iteration* to solve for an (approximate) solution to the Bellman equation.

In [ ]:
planner = DynamicProgrammingPlanner(
    problem,
    grid=grid,
    options=DynamicProgrammingOptions(
        alpha=1.0,
        tol=0.1,
        max_iterations=2000,
        out_of_bound_cost=INF,
        verbose=True,
    ),
)

result = planner.solve().policy
planner.clean_infeasible_set()

## Cost-to-go $J^*$

The following figures illustrate the computed optimal cost-to-go $J^*$ for every starting state. Note that the plotted $J^*$ is saturated at the out-of-domain penalty to better show the range of interest.

In [ ]:
plotting.plot_value(grid, result.J, vmax=INF)
plotting.plot_value_3d(grid, np.clip(result.J, 0.0, INF))

## Showing the computed control law

The next figure shows a map illustrating the computed optimal torque to apply as a function of the two system states:

$$\tau = \pi^*(\theta, \dot\theta)$$

In [ ]:
plotting.plot_policy(grid, result.pi)

controller = result.controller()  # interpolated look-up table controller

## Simulation

Here we show the control law in action, with a closed-loop trajectory starting at the state $[\theta = 0, \dot\theta = 0]$ (the pendulum hanging down).

In [ ]:
plant.x0 = np.array([0.0, 0.0])  # initial state
tf = 10.0  # simulation time

cl_sys = DiagramSystem()
cl_sys.add_subsystem(controller, "ctl")
cl_sys.add_subsystem(plant, "plant")
cl_sys.connect("plant", "y", "ctl", "x")
cl_sys.connect("ctl", "u", "plant", "u")
cl_sys.name = "Pendulum with DP controller"

traj = cl_sys.compute_trajectory(tf=tf, n_steps=2001)
cl_sys.plot_trajectory(traj)

## Animation of the simulation

Here the following function generates an animation of the computed trajectory.

In [ ]:
cl_sys.camera_scale = 2.0
cl_sys.animate(traj)

## Phase-plane trajectory

Here the same trajectory is shown on the phase plane of the pendulum. The vector field illustrates the natural dynamics along which the pendulum would evolve if no torque were applied on the system.

In [ ]:
plant.plot_phase_plane(traj)

## Performance

Here the performance, in terms of the defined cost function $J = \int g(x,u,t)\,dt$, is shown. Note that $\dot J = g(x,u,t)$ is the increment of cost at each instant and $J$ is the cumulative cost.

In [ ]:
# Rebuild the applied torque from the control law, then evaluate the cost
u_sim = np.array([controller.action(x) for x in traj.x.T]).T

plant_traj = Trajectory(t=traj.t, x=traj.x, u=u_sim)
plant_traj = cf.evaluate_trajectory(plant_traj)

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
axes[0].plot(plant_traj.t, plant_traj.signals["cost_rate"][0])
axes[0].set_ylabel("$\\dot{J} = g(x,u,t)$")
axes[0].grid(True, alpha=0.3)
axes[1].plot(plant_traj.t, plant_traj.signals["cost"][0])
axes[1].set_ylabel("$J = \\int g \\, dt$")
axes[1].set_xlabel("t [s]")
axes[1].grid(True, alpha=0.3)
plt.show()

print("Total trajectory cost J =", round(float(plant_traj.signals["cost"][0, -1]), 1))

## Notes and hints

- **Task b):** go back to the cost-function parameters cell and increase `cf.Q[0, 0]` (position error weight), then re-run all the following cells.
- **Task c):** for a minimum-time behaviour, try a constant running cost $g = 1$ away from the target with `cf.ontarget_check = True` (zero cost inside the target zone `EPS`). Minilink also ships a ready-made minimum-time cost: `from minilink.core.costs import TimeCost` then `cf = TimeCost.from_system(plant, xbar=np.array([-np.pi, 0.0]), eps=0.2)`.
- For several reasons, the algorithm may have difficulty converging for some cost functions. Try minor changes if that happens: the goal here is not to make you tune the convergence parameters (a topic not yet covered).